# Convolution and Pooling

A dense network flattens an image into one long list of pixels, so it does not know which pixels are neighbours. A **convolutional neural network** (CNN) keeps the image as a grid and looks at small neighbourhoods of pixels at a time. In this notebook you compute the two operations a CNN is built from, convolution and pooling, by hand and with NumPy, before Keras does them for you in notebook 02.

## Learning Objectives

By the end of this notebook, you should be able to:

- Calculate the convolution of an image with a filter, by hand and with NumPy.
- Calculate the output size of a convolution from the image size, the filter size, the padding and the stride.
- Apply edge-detection filters and max pooling to a real image.
- Calculate the number of parameters of a convolutional layer.

## Setup

In [ ]:
import keras
import matplotlib.pyplot as plt
import numpy as np

## 1. An Image Is an Array of Numbers

A greyscale image is a 2D array: one number per pixel, from 0 (black) to 255 (white). The [MNIST](https://keras.io/api/datasets/mnist/) dataset contains 70,000 handwritten digits of 28 × 28 pixels. [`keras.datasets.mnist.load_data()`](https://keras.io/api/datasets/mnist/#load_data-function) downloads it on the first run (about 11 MB) and returns a training and a test set; here you use only one digit.

In [ ]:
(X_train, y_train), _ = keras.datasets.mnist.load_data()
digit = X_train[0]
print(f"Shape: {digit.shape}, data type: {digit.dtype}, values from {digit.min()} to {digit.max()}, label: {y_train[0]}")

fig, (ax_image, ax_values) = plt.subplots(1, 2, figsize=(11, 5))
ax_image.imshow(digit, cmap="gray")
ax_image.set_title("The digit as an image")
ax_values.imshow(digit[4:14, 4:14], cmap="gray", extent=(3.5, 13.5, 13.5, 3.5))  # axes show the original rows and columns
for row in range(4, 14):
    for col in range(4, 14):
        value = digit[row, col]
        ax_values.text(col, row, value, ha="center", va="center", fontsize=7, color="red" if value < 128 else "black")
ax_values.set_title("Rows and columns 4 to 13 as numbers")
plt.show()

The digit is a 5. Neighbouring pixels belong together: the bright pixels form the lines of the handwritten digit, and the shape of these lines is what makes a 5 a 5. A dense network receives the 784 values as one flat list, so it would treat two pixels next to each other exactly like two pixels at opposite corners. A convolution uses the neighbourhood directly.

## 2. Convolution

A **filter** (also called a **kernel**) is a small grid of numbers, here 3 × 3. A **convolution** slides the filter over the image. At every position it multiplies each filter value with the pixel underneath, adds the nine products up and writes the sum into a new image, the **feature map**.

The figure applies a filter to a simple 6 × 6 image of a face: two eyes, the corners of the mouth and the mouth. The filter has positive values in its middle row and negative values above and below, so it gives a large sum where the image has a horizontal line with empty pixels above and below it.

<img src="assets/face_convolution.png" width="900">

The window outlined in amber covers rows 3 to 5 and columns 1 to 3 of the image. Only its middle row contains ones, so the sum is $3 \times 2 = 6$: one value of the feature map. The feature map is largest along the mouth (6), smaller at the eyes (2) and negative where the image is empty in the middle row.

**Your turn:** compute the value in the top-left corner of the feature map by hand, from rows 0 to 2 and columns 0 to 2 of the image, before you read on.

<details>
<summary>Show the answer</summary>

Only one pixel of the window is 1: row 1, column 1, which lies under the middle row of the filter (+2). All other pixels are 0, so the sum is $1 \cdot 2 = 2$, the value in the figure. The window covers one eye.

</details>

In [ ]:
face = np.array(
    [
        [0, 0, 0, 0, 0, 0],
        [0, 1, 0, 0, 1, 0],
        [0, 0, 0, 0, 0, 0],
        [1, 0, 0, 0, 0, 1],
        [0, 1, 1, 1, 1, 0],
        [0, 0, 0, 0, 0, 0],
    ]
)
line_filter = np.array([[-1, -1, -1], [2, 2, 2], [-1, -1, -1]])

The function below performs a convolution with two loops, one over the rows and one over the columns of the feature map. An $n \times n$ image and an $f \times f$ filter give a feature map of $(n - f + 1) \times (n - f + 1)$, because the filter has to fit completely inside the image. Complete the line that computes one value: the sum of the element-wise product of the window `image[row:row + f, col:col + f]` and the filter. Complete it before you continue: the rest of the notebook uses `convolve`.

In [ ]:
def convolve(image, kernel):
    """Return the feature map of a square image convolved with a square kernel (no padding, stride 1)."""
    n, f = image.shape[0], kernel.shape[0]
    feature_map = np.zeros((n - f + 1, n - f + 1))
    for row in range(n - f + 1):
        for col in range(n - f + 1):
            # TODO: multiply the window of the image element-wise with the kernel and add up the products
            feature_map[row, col] = ...
    return feature_map


face_map = convolve(face, line_filter)
face_map

The result is the feature map from the figure. A CNN passes every feature map through an activation function, usually ReLU, which sets the negative values to 0 and keeps the positive ones; [`np.maximum`](https://numpy.org/doc/stable/reference/generated/numpy.maximum.html) applies it element by element. After ReLU only the places where the filter found a horizontal line remain:

In [ ]:
face_map_relu = np.maximum(0, face_map)
face_map_relu

Deep learning libraries call this operation a convolution, although in mathematics it is called a cross-correlation (a mathematical convolution flips the filter first). Because the filter values are learnt, the difference does not matter for a neural network.

## 3. Padding and Stride

The feature map is smaller than the image: a 6 × 6 image gives a 4 × 4 map, and the pixels at the border are covered by fewer windows than the pixels in the middle. Two settings change this:

- **Padding** adds a ring of zeros around the image before the convolution. With a 3 × 3 filter, a padding of 1 keeps the output the same size as the input; a 5 × 5 filter needs a padding of 2. In Keras, `padding="same"` adds as many zeros as the filter needs to keep the size, and `padding="valid"` adds none. Padding serves two purposes: the border pixels are covered by as many windows as the pixels in the middle, so a feature at the edge of the image is not lost, and the feature maps keep their size, so you can stack many convolutional layers without the feature maps becoming too small to work with.
- **Stride** is the number of pixels the window moves at each step. A stride of 2 skips every other position and roughly halves the size of the output. Its purpose is to shrink the feature maps on purpose: the next layers have fewer values to process, which saves computation and memory. Pooling (section 5) is the other common way to shrink them; most networks use a stride of 1 in their convolutions and shrink with pooling.

The figure shows both on a 5 × 5 image:

<img src="assets/padding_stride.png" width="900">

For an $n \times n$ image, an $f \times f$ filter, a padding of $p$ and a stride of $s$, the output has the size

$$\left\lfloor \frac{n + 2p - f}{s} \right\rfloor + 1,$$

where $\lfloor \cdot \rfloor$ rounds down: a window that does not fit completely is left out. [`np.pad`](https://numpy.org/doc/stable/reference/generated/numpy.pad.html) adds the padding, so the face keeps its size of 6 × 6:

In [ ]:
def output_size(n, f, padding=0, stride=1):
    """Return the side length of the output of a convolution."""
    return (n + 2 * padding - f) // stride + 1


padded_face = np.pad(face, 1)  # one ring of zeros on every side: 8 x 8
print("Padded image:", padded_face.shape, " feature map:", convolve(padded_face, line_filter).shape)
print("Formula, padding 1:", output_size(6, 3, padding=1))
print("Formula, MNIST digit, 3 x 3 filter, no padding:", output_size(28, 3))
print("Formula, MNIST digit, 3 x 3 filter, stride 2:", output_size(28, 3, stride=2))

## 4. Filters That Detect Edges

The filter on the face was chosen to find one pattern. Image processing uses filters designed for common patterns. The **Sobel filters** respond to edges: the first one to changes in brightness from left to right (vertical edges), the second, its transpose, to changes from top to bottom (horizontal edges). You can try these and other filters interactively on [Image Kernels explained visually](https://setosa.io/ev/image-kernels/).

The cell below scales the digit to values between 0 and 1 and convolves it with both Sobel filters.

In [ ]:
sobel_vertical = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]])
sobel_horizontal = sobel_vertical.T  # the transpose detects horizontal edges

digit_scaled = digit / 255
vertical_edges = convolve(digit_scaled, sobel_vertical)
horizontal_edges = convolve(digit_scaled, sobel_horizontal)
print("Feature map shape:", vertical_edges.shape)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
for ax, image, title in zip(
    axes,
    [digit_scaled, vertical_edges, horizontal_edges],
    ["Digit", "Sobel filter for vertical edges", "Sobel filter for horizontal edges"],
):
    shown = ax.imshow(image, cmap="gray" if title == "Digit" else "RdBu_r")
    ax.set_title(title)
    ax.axis("off")
    fig.colorbar(shown, ax=ax, fraction=0.046)
plt.show()

Each feature map has 26 × 26 values, as the formula predicts for a 28 × 28 image and a 3 × 3 filter. Red marks large positive values and blue large negative ones. The vertical-edge filter responds to the lines that run from top to bottom, such as the left side of the 5 and its curve: red on the left edge of a line, where the brightness increases from left to right, and blue on the right edge, where it decreases. The horizontal-edge filter responds most strongly to the horizontal line at the top and to the bottom of the curve. Where the image is uniform, inside a line or in the black background, both maps are close to 0.

## 5. Pooling

A **pooling** layer shrinks a feature map by summarising small windows. **Max pooling** with a 2 × 2 window and a stride of 2 keeps only the largest value of each window, so it halves the height and the width. The strongest responses survive, and a feature that moves by one pixel usually still ends up in the same window, which makes the result less sensitive to small shifts. **Average pooling** keeps the mean of each window instead.

<img src="assets/max_pooling.png" width="600">

Pooling has no padding, and its stride equals its window size, so the output-size formula of section 3 becomes $\lfloor (n - 2)/2 \rfloor + 1 = \lfloor n/2 \rfloor$ for a 2 × 2 window. For an even size this is exactly half: 26 becomes 13. For an odd size the last row and column do not fill a window and are dropped: 11 becomes 5, not 5.5.

The function below reshapes the feature map so that every 2 × 2 window gets its own two axes, then takes the maximum over them.

In [ ]:
def max_pool(feature_map, size=2):
    """Return the max pooling of a square feature map with a size x size window and a stride of size."""
    n = feature_map.shape[0] // size * size  # drop a last row and column that do not fill a window
    windows = feature_map[:n, :n].reshape(n // size, size, n // size, size)
    return windows.max(axis=(1, 3))


print(max_pool(face_map_relu))

pooled_edges = max_pool(np.maximum(0, vertical_edges))
print("Vertical edges after ReLU and pooling:", pooled_edges.shape)
plt.imshow(pooled_edges, cmap="gray")
plt.title("Vertical edges after ReLU and max pooling")
plt.axis("off")
plt.show()

Max pooling the face feature map after ReLU gives `[[2, 2], [6, 6]]`, the result in the figure: the mouth and the eyes are still there, in a map a quarter of the size. Pooling the vertical edges of the digit turns 26 × 26 values into 13 × 13. The image is coarser, but the outline of the 5 is still recognisable, with four times fewer values.

### Small Shifts: What Convolution, Pooling and Padding Do

A face should be recognised whether it is in the middle of a photo or a few pixels to the left. Three parts of a CNN work together to make this possible:

- **Convolution moves with the input.** The same filter is applied at every position, so when the feature moves by one pixel, its response in the feature map moves by one cell. The feature is still found, only somewhere else. This property is called **translation equivariance**.
- **Pooling reduces the effect of small shifts.** Max pooling keeps the largest value of each window, wherever it lies inside the window. A shift that stays inside one pooling window therefore gives exactly the same pooled output. A shift across a window boundary still moves the output, so the network is only **approximately invariant** to small shifts. Every further pooling layer makes the windows cover a larger part of the image, so deeper networks tolerate larger shifts.
- **Padding keeps the border.** Without padding, the centre of the filter never reaches the outermost pixels, so a small feature at the very edge can be missed. With padding the filter can centre on every pixel, and the edge behaves like the middle.

The figure shows all three with an "eye", a single bright pixel, and a filter that responds to such a dot:

<img src="assets/shift_invariance.png" width="900">

The cell below reproduces the figure with your `convolve` and `max_pool` functions.

In [ ]:
dot_filter = np.array([[-1, -1, -1], [-1, 8, -1], [-1, -1, -1]])  # responds to a single bright pixel

for col in [1, 2, 3]:
    eye_image = np.zeros((8, 8))
    eye_image[3, col] = 1  # the eye in row 3
    eye_map = np.maximum(0, convolve(eye_image, dot_filter))
    pooled = max_pool(eye_map)
    print(f"Eye at column {col}: feature map peak at {np.argwhere(eye_map == 8).tolist()}, "
          f"pooled peak at {np.argwhere(pooled == 8).tolist()}")

border_image = np.zeros((8, 8))
border_image[3, 0] = 1  # the eye on the left edge
print("Eye at the border, no padding: maximum", np.maximum(0, convolve(border_image, dot_filter)).max())
print("Eye at the border, padding 1:  maximum", np.maximum(0, convolve(np.pad(border_image, 1), dot_filter)).max())

## 6. From Filters to a Convolutional Layer

In this notebook you chose the filter values. In a CNN, the filter values are **weights**: they start random and are learnt with backpropagation, like the weights of a dense layer. The network therefore finds the patterns that help with the task itself, edges and line endings in the first layer and combinations of them in later layers.

The figure shows this idea for a network that recognises faces. The filters of the first layer respond to simple edges. A filter in the second layer works on the feature maps of the first layer, so it responds where several edges appear together in a particular arrangement: a horizontal edge with a curve above it forms an eye. A filter in the third layer combines the parts in the same way into a whole face.

<img src="assets/feature_hierarchy.png" width="1000">

Each layer also covers a larger area of the image than the one before. A 3 × 3 filter in the first layer covers 3 × 3 pixels. After a 2 × 2 pooling, one value of the feature map summarises a 2 × 2 area, so a 3 × 3 filter in the second layer already covers about 8 × 8 pixels of the original image. Deeper layers therefore detect larger structures. The figure is a simplified drawing: the features of real networks are less regular, but follow the same progression from edges to parts to objects (Zeiler and Fergus, 2014).

### Colour Images and Feature Maps as a Box

A greyscale image is a 2D grid. A colour image has three grids of the same size, one each for the red, green and blue intensity of every pixel, so it is a 3D array of height × width × 3. The three grids are called **channels**.

A filter for a colour image has the same depth as the image: a 3 × 3 filter becomes 3 × 3 × 3, one 3 × 3 slice per channel. Panel A of the figure shows how it is applied. Each channel is convolved with its own slice, exactly as in section 2, and the three results are added up, together with one bias, into a **single** feature map. Then the activation function, usually ReLU, is applied to every value of this map, as in section 2. The order in every convolutional layer is: convolve each channel with its slice, add up, add the bias, apply the activation. Pooling, if any, follows in a separate layer. In Keras, `Conv2D(..., activation="relu")` performs all of these steps inside one layer. The filter can therefore combine colours: for example, respond to an edge only where red meets green.

A convolutional layer has several filters, and each filter gives one feature map. Keras stacks them, so the output of the layer is again a box, of height × width × number of filters (panel B). Each slice of this box is the feature map of one filter, and the depth is called the number of **channels** of the output, like the colour channels of the input. The next convolutional layer treats this box exactly like a colour image with more channels: its filters have one slice for each of them.

<img src="assets/colour_convolution.png" width="1000">

The cell below checks the shapes with random numbers (the bias is left out). Keras stores the filters of a layer in the same order: height, width, input channels, filters.

In [ ]:
rng = np.random.default_rng(0)
colour_image = rng.random((6, 6, 3))  # height, width, 3 colour channels
one_filter = rng.normal(size=(3, 3, 3))  # one 3 x 3 slice per channel

# convolve each channel with its own slice, add up the three results, then apply ReLU
feature_map = np.maximum(0, sum(convolve(colour_image[:, :, c], one_filter[:, :, c]) for c in range(3)))
print("Image", colour_image.shape, "* one filter", one_filter.shape, "-> one feature map", feature_map.shape)

filters = rng.normal(size=(3, 3, 3, 4))  # 4 filters: height, width, input channels, filters
maps = [np.maximum(0, sum(convolve(colour_image[:, :, c], filters[:, :, c, k]) for c in range(3))) for k in range(4)]
output_box = np.stack(maps, axis=-1)  # stack the 4 feature maps along a new last axis
print("Image", colour_image.shape, "* 4 filters", filters.shape, "-> output box", output_box.shape)

### Channels and Parameters

Each filter has $f \cdot f$ weights for each of the $c$ input channels and one bias. A layer with $k$ filters of size $f \times f$ on an input with $c$ channels therefore has

$$(f \cdot f \cdot c + 1) \cdot k$$

parameters. This number does not depend on the size of the image, because the same filter is used at every position. A dense layer, in contrast, needs one weight per input pixel and neuron.

Complete the `return` line with this formula, then check your function on the examples below: the first two are the convolutional layers of the network you build in notebook 02, so `model.summary()` will show the same numbers there.

In [ ]:
def conv_parameters(f, channels, filters):
    """Return the number of weights and biases of a convolutional layer."""
    # TODO: the weights of k filters of size f x f x channels, plus one bias per filter
    return ...


print("6 filters of 5 x 5 on a greyscale image:      ", conv_parameters(5, 1, 6))
print("16 filters of 5 x 5 on 6 input channels:      ", conv_parameters(5, 6, 16))
print("32 filters of 3 x 3 on a colour image:         ", conv_parameters(3, 3, 32))
print("Dense layer, 128 neurons on 28 x 28 pixels:    ", 28 * 28 * 128 + 128)

The first two lines are the two convolutional layers of the network you build in notebook 02: 6 filters of 5 × 5 on a greyscale image need 156 parameters, and 16 filters of 5 × 5 on the 6 feature maps of the first layer need 2,416. Even 32 filters on a colour image need only 896. A single dense layer of 128 neurons on the same 28 × 28 image needs 100,480 parameters, because every pixel has its own weight for every neuron.

## Try It: Add a Stride

`convolve` always moves the window by one pixel. Write `convolve_strided`, which moves it by `stride` pixels. Two things change: the size of the feature map, which you can take from `output_size`, and the position of the window, which starts at `row * stride` and `col * stride`. Convolve the digit with the vertical Sobel filter and a stride of 2, and check that the shape matches the formula.

In [ ]:
def convolve_strided(image, kernel, stride=1):
    """Return the feature map of a square image convolved with a square kernel and the given stride."""
    n, f = image.shape[0], kernel.shape[0]
    # TODO: the side length of the feature map, from output_size
    size = ...
    feature_map = np.zeros((size, size))
    for row in range(size):
        for col in range(size):
            # TODO: the top-left corner of the window moves by stride pixels at every step
            top, left = ..., ...
            feature_map[row, col] = np.sum(image[top:top + f, left:left + f] * kernel)
    return feature_map


strided_edges = convolve_strided(digit_scaled, sobel_vertical, stride=2)
print("Shape:", strided_edges.shape, " formula:", output_size(28, 3, stride=2))
print("Same as every second value of the stride-1 map:", np.allclose(strided_edges, vertical_edges[::2, ::2]))

## Summary

- An image is a grid of numbers. A convolution slides a small filter over it and writes the sum of the element-wise products into a feature map; large values mark the places where the image matches the pattern of the filter.
- An $n \times n$ image and an $f \times f$ filter with padding $p$ and stride $s$ give an output of $\lfloor (n + 2p - f)/s \rfloor + 1$. Padding keeps the size; a larger stride shrinks it.
- ReLU keeps only the positive responses, and max pooling halves the size while keeping the strongest response of each window.
- A shifted feature gives a shifted feature map (equivariance); pooling makes the output the same for small shifts inside a window (approximate invariance); padding lets features at the border be detected.
- A colour image has three channels. A filter has one slice per input channel and adds their results and its bias into one feature map, which then passes through the activation function; a layer with $k$ filters outputs a box of height × width × $k$.
- In a CNN the filter values are learnt weights. A convolutional layer with $k$ filters of size $f \times f$ on $c$ input channels has $(f \cdot f \cdot c + 1) \cdot k$ parameters, independent of the image size.

Next, continue with [02 - A Convolutional Network with Keras](02_cnn_with_keras.ipynb).

## References

- Dumoulin, V. and Visin, F. (2016). [A guide to convolution arithmetic for deep learning](https://arxiv.org/abs/1603.07285): padding, stride and output sizes, with many figures.
- CS231n. [Convolutional Neural Networks](https://cs231n.github.io/convolutional-networks/): convolutional and pooling layers, and how their sizes and parameters are calculated.
- Zeiler, M. D. and Fergus, R. (2014). [Visualizing and Understanding Convolutional Networks](https://arxiv.org/abs/1311.2901): what the filters of each layer of a real CNN respond to, from edges to object parts.
- Powell, V. [Image Kernels explained visually](https://setosa.io/ev/image-kernels/): try blur, sharpen and edge filters on a photo in the browser.
- NumPy API: [`np.pad`](https://numpy.org/doc/stable/reference/generated/numpy.pad.html).
- The structure follows the convolutional neural networks notebook of the Spiced Academy deep learning course.